# Recency weighting для tuned XGBoost

Повторяем controlled experiment LightGBM для secondary Pilot v4 XGBoost. Проверяем, помогает ли более высокий `sample_weight` свежих наблюдений при неизменных признаках, параметрах и обработке категорий. Это изменение loss, а не новый feature. Гипотеза разумна при изменении пассажиропотока со временем, но её нужно подтвердить на нескольких временных срезах. Nov–Dec не участвует в оценке качества.

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/xgboost_recency_weighting_experiment.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(root / 'ml/src'))
import model_selection as ms
import recency_weighting_experiment as rw
import xgboost_recency_weighting_experiment as xrw

rows, folds, hpo = rw.load_inputs('XGBRegressor')
parameters = rw.model_contract(hpo, 'XGBRegressor')
display(pd.DataFrame([{'historical_rows': len(rows), 'target': 'boardings',
                       'feature_count': len(ms.FEATURE_CONFIGURATIONS[rw.CONFIGURATION]),
                       'fold_count': len(folds), 'model': 'XGBRegressor'}]))

,historical_rows,target,feature_count,fold_count,model
0,72960,boardings,12,3,XGBRegressor


## Frozen contract и temporal validation

Используем `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES` и tuned XGBoost из HPO: `n_estimators=500`, `max_depth=6`, `learning_rate=0.1`, `min_child_weight=1`, остальные параметры — из model-selection contract. Категории задаются фиксированными pandas categories; XGBoost применяет native categorical splits. Train каждого fold содержит только прошлое: Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct. Случайное разбиение здесь нарушило бы порядок информации. Ни один вариант не подбираем специально под Sep–Oct.

In [2]:
reference = rw.evaluate_configuration(folds, 'reference', parameters, 'XGBRegressor')
rw.assert_reference_reproduced(reference, hpo, 'XGBRegressor')
display(pd.DataFrame([{'fold': spec[0], 'wape': result['wape'],
                       'absolute_error': result['absolute_error']}
                      for spec, result in zip(ms.FOLDS, reference, strict=True)]).round(6))
print('Frozen tuned XGBoost reference reproduced exactly.')

,fold,wape,absolute_error
0,jan_apr_to_may_jun,0.153758,1776814
1,jan_jun_to_jul_aug,0.198392,2126301
2,jan_aug_to_sep_oct,0.115491,1473023


Frozen tuned XGBoost reference reproduced exactly.


## Четыре заранее заданные схемы весов

`reference` не задаёт веса. `fixed` использует ступени 1.00 / 0.95 / 0.85 / 0.70 для возрастов 0–30 / 31–60 / 61–120 / более 120 дней. `exponential_120` и `exponential_180` убывают плавно: half-life — срок, за который вес уменьшается вдвое. Проверяем только 120 и 180 дней, уже заданные в LightGBM experiment; новых значений не ищем. Для каждого fold возраст считается заново относительно последней даты его train. Семантика features, target, XGBoost preprocessing, postprocessing и route 5 policy одинаковы во всех четырёх runs.

In [3]:
all_folds = {'reference': reference}
for name in rw.WEIGHTING_CONFIGURATIONS[1:]:
    all_folds[name] = rw.evaluate_configuration(folds, name, parameters, 'XGBRegressor')
result = rw.build_result(all_folds, hpo, 'XGBRegressor')
xrw.ARTIFACT.write_text(__import__('json').dumps(result, ensure_ascii=False, indent=2,
                                                  allow_nan=False) + '\n', encoding='utf-8')
summaries = result['summaries']
reference_wape = summaries['reference']['pooled_wape']
table = pd.DataFrame([{
    'Weighting': name,
    'May-Jun': all_folds[name][0]['wape'],
    'Jul-Aug': all_folds[name][1]['wape'],
    'Sep-Oct': all_folds[name][2]['wape'],
    'Pooled WAPE': summaries[name]['pooled_wape'],
    'Delta vs reference': summaries[name]['pooled_wape'] - reference_wape,
    'Pooled AE': summaries[name]['pooled_absolute_error'],
    'WAPE-score': summaries[name]['wape_score'],
    'Mean WAPE': summaries[name]['mean_wape'],
    'Median WAPE': summaries[name]['median_wape'],
    'Fit seconds': summaries[name]['train_seconds'],
    'Predict seconds': summaries[name]['inference_seconds'],
} for name in rw.WEIGHTING_CONFIGURATIONS])
display(table.round(6))

,Weighting,May-Jun,Jul-Aug,Sep-Oct,Pooled WAPE,Delta vs reference,Pooled AE,WAPE-score,Mean WAPE,Median WAPE,Fit seconds,Predict seconds
0,reference,0.153758,0.198392,0.115491,0.153481,0.000000,5376138,0.846519,0.155880,0.153758,29.968260,0.952280
1,fixed,0.153834,0.196043,0.116568,0.153180,-0.000301,5365583,0.846820,0.155482,0.153834,29.874314,0.944069
2,exponential_120,0.157525,0.195789,0.118097,0.154876,0.001395,5425008,0.845124,0.157137,0.157525,30.809483,0.964549
3,exponential_180,0.156928,0.198218,0.118587,0.155601,0.002120,5450387,0.844399,0.157911,0.156928,29.743605,0.987632


## Устойчивость и маршруты

Основной критерий — pooled WAPE: сумма абсолютных ошибок трёх holdouts, делённая на сумму фактических посадок. Fold WAPE показывает качество каждого периода; WAPE-score равен `max(0, 1 - WAPE)`. Улучшение только одного fold недостаточно для поддержки гипотезы. Также проверяем pooled WAPE по маршрутам: route 5 имеет нулевой target и принудительно нулевой prediction, поэтому его WAPE не определён. `SUPPORTED` требует улучшения pooled WAPE без ухудшения folds и сильного route degradation; `MIXED` означает улучшение pooled при неоднородном эффекте; `REJECTED` — отсутствие улучшения pooled.

In [4]:
best = result['best_weighting']
route_rows = []
for route in ms.ROUTES:
    base = summaries['reference']['per_route'][str(route)]['wape']
    candidate = summaries[best]['per_route'][str(route)]['wape']
    route_rows.append({'route': route, 'reference': base, 'best_weighting': candidate,
                       'delta': candidate - base if base is not None else None})
display(pd.DataFrame(route_rows).round(6))
display(pd.DataFrame([{'best_weighting': best,
                       'delta_pooled_wape': result['best_delta_pooled_wape'],
                       'classification': result['classification']}]).round(6))
display(pd.DataFrame([{'fold': name, 'delta_wape': delta}
                      for name, delta in result['best_fold_delta_wape'].items()]).round(6))

,route,reference,best_weighting,delta
0,1,0.122247,0.122362,0.000115
1,5,NaN,NaN,NaN
2,7,0.270156,0.269147,-0.001009
3,11,0.125528,0.124857,-0.000671
4,12,0.118391,0.118533,0.000141
5,17,0.113507,0.114120,0.000612
6,25,0.196022,0.191482,-0.004540
7,26,0.168142,0.169876,0.001733
8,28,0.184438,0.180787,-0.003651
9,50,0.215810,0.214864,-0.000946


,best_weighting,delta_pooled_wape,classification
0,fixed,-0.000301,RECENCY_WEIGHTING_MIXED


,fold,delta_wape
0,jan_apr_to_may_jun,0.000076
1,jan_jun_to_jul_aug,-0.002349
2,jan_aug_to_sep_oct,0.001078


## Отдельный Jan–Oct fit варианта D

Как и для LightGBM, дополнительно обучаем именно вариант D (`exponential_180`) на всей доступной Jan–Oct истории. Это отдельный исследовательский submission независимо от классификации temporal experiment. Sep–Oct теперь часть известного train, а веса отсчитываются от 2025-10-31. Nov–Dec не используется ни для выбора весов, ни для оценки accuracy. Primary LightGBM и unweighted XGBoost submission не перезаписываются.

In [5]:
assert len(rows) == 72_960 and rows['date'].max() == pd.Timestamp('2025-10-31')
weights, weight_info = rw.sample_weights(rows, 'exponential_180')
assert weight_info['origin'] == '2025-10-31'
assert len(weights) == len(rows) and np.isfinite(weights).all()
x_train = ms.make_features(rows, rw.CONFIGURATION, 'XGBRegressor')
assert x_train.columns.tolist() == list(ms.FEATURE_CONFIGURATIONS[rw.CONFIGURATION])
model_d = XGBRegressor(**parameters)
start = perf_counter()
model_d.fit(x_train, rows['boardings'], sample_weight=weights)
fit_seconds = perf_counter() - start
display(pd.DataFrame([{'train_rows': len(rows), 'weight_origin': weight_info['origin'],
                       'weight_min': weights.min(), 'weight_mean': weights.mean(),
                       'weight_max': weights.max(), 'fit_seconds': fit_seconds}]).round(3))

,train_rows,weight_origin,weight_min,weight_mean,weight_max,fit_seconds
0,72960,2025-10-31,0.311,0.59,1.0,17.642


## Nov–Dec inference и проверка submission

Forecast читаем напрямую из model-ready CSV в исходном порядке. После XGBoost prediction применяем тот же postprocessing, что в HPO: `floor(max(raw, 0) + 0.5)`, затем route 5 обнуляем. Сохраняем `route;date;hour;prediction` в отдельный файл. После записи перечитываем его и сверяем с forecast grid и organizer template; эти проверки не говорят о точности Nov–Dec.

In [6]:
forecast = pd.read_csv(root / 'data/processed/model_ready_forecast_2025_nov_dec.csv',
                       sep=';', parse_dates=['date'],
                       dtype={'temperature_bucket': ms.BUCKET_DTYPE})
keys = ['route', 'date', 'hour']
assert len(forecast) == 14_640 and 'boardings' not in forecast.columns
assert not forecast.isna().any().any() and not forecast.duplicated(keys).any()
assert set(forecast['route']) == set(ms.ROUTES)
assert set(forecast['hour']) == set(range(24))
assert set(forecast['date']) == set(pd.date_range('2025-11-01', '2025-12-31'))
x_forecast = ms.make_features(forecast, rw.CONFIGURATION, 'XGBRegressor')
assert x_forecast.columns.tolist() == x_train.columns.tolist()
start = perf_counter()
raw_prediction = model_d.predict(x_forecast)
predict_seconds = perf_counter() - start
assert np.isfinite(raw_prediction).all()
prediction = np.floor(np.maximum(raw_prediction, 0) + 0.5).astype('int64')
prediction[forecast['route'].eq(5).to_numpy()] = 0
submission = forecast[keys].copy()
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission['prediction'] = prediction
submission_path = root / 'data/submissions/pilot_model_v4_xgboost_recency_exp180_submission.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, sep=';', index=False, encoding='utf-8')
saved = pd.read_csv(submission_path, sep=';')
template = pd.read_csv(root / 'data/raw/test_submission.csv', sep=';')
assert saved.columns.tolist() == ['route', 'date', 'hour', 'prediction']
assert len(saved) == len(forecast) == len(template) == 14_640
assert saved[keys].equals(submission[keys]) and saved[keys].equals(template[keys])
assert not saved.duplicated(keys).any() and not saved.isna().any().any()
assert saved['route'].nunique() == 10 and saved['hour'].nunique() == 24
assert saved['date'].min() == '2025-11-01' and saved['date'].max() == '2025-12-31'
assert pd.api.types.is_integer_dtype(saved['prediction'])
assert saved['prediction'].ge(0).all() and np.isfinite(saved['prediction'].to_numpy()).all()
assert saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all()
assert saved.equals(submission)
display(pd.DataFrame([{'file': str(submission_path.relative_to(root)), 'rows': len(saved),
                       'total': int(saved['prediction'].sum()),
                       'zero_count': int(saved['prediction'].eq(0).sum()),
                       'median': float(saved['prediction'].median()),
                       'p95': float(saved['prediction'].quantile(.95)),
                       'p99': float(saved['prediction'].quantile(.99)),
                       'max': int(saved['prediction'].max()),
                       'predict_seconds': predict_seconds, 'checks_passed': True}]).round(3))
display(saved.assign(month=saved['date'].str[:7]).groupby('month', as_index=False)['prediction'].sum())

,file,rows,total,zero_count,median,p95,p99,max,predict_seconds,checks_passed
0,data\submissions\pilot_model_v4_xgboost_recenc...,14640,12602030,2852,501.0,2961.3,3953.05,5316,0.308,True


,month,prediction
0,2025-11,6121622
1,2025-12,6480408


## Вывод

Лучший weighted-вариант для XGBoost — fixed: pooled WAPE `0.153180` против reference `0.153481` (Δ `−0.000301`). Он улучшил Jul–Aug на `−0.002349`, но ухудшил May–Jun на `+0.000076` и Sep–Oct на `+0.001078`. Максимальное ухудшение pooled route WAPE — route 26 (`+0.001733`); сильного route degradation нет, но эффект между folds неустойчив. Классификация — **`RECENCY_WEIGHTING_MIXED`**. Отдельный Nov–Dec submission варианта D создан по запросу на аналог LightGBM: D проиграл reference по pooled WAPE (`0.155601` против `0.153481`), поэтому это только исследовательский inference artifact, а не рекомендация заменить secondary XGBoost. Primary Pilot v4 и accepted feature registry остаются прежними.